# Mapa de Problemas em Ciclofaixas do Rio de Janeiro

**Objetivo:** Criar um mapa interativo onde qualquer pessoa pode **clicar num local e reportar um problema** na infraestrutura cicloviária.

### O que este notebook faz

1. Baixa dados de ciclovias do Rio de Janeiro (OpenStreetMap)
2. Gera um mapa interativo com camadas por tipo de ciclovia
3. **Analisa gaps** (descontinuidades) na malha cicloviária
4. **Adiciona a funcionalidade de reportar problemas:**
   - Clique no mapa para marcar um local com problema
   - Preencha: categoria, descrição, severidade
   - Anexe fotos (múltiplas)
   - Exporte/importe os dados como JSON

### O que você vai aprender

| Conceito | Seção | Dificuldade |
|---|---|---|
| Consultar dados geoespaciais no OpenStreetMap | 2. Download | Básico |
| Explorar e classificar dados com GeoDataFrame | 3. Exploração | Básico |
| Detectar descontinuidades numa rede viária (gaps) | 4. Gaps | Intermediário |
| Criar mapas interativos com Folium | 6. Gerar mapa | Básico |
| Injetar JavaScript num mapa HTML | 5. Reportar | Avançado |
| Persistência de dados no navegador (localStorage) | 5. Reportar | Intermediário |

### Pré-requisitos

- **Python básico** — saber rodar células, entender variáveis, loops, funções
- **Pandas** — desejável mas não obrigatório (explicamos o que cada operação faz)
- **Nenhum conhecimento prévio** de dados geoespaciais, OSM ou Folium necessário

### Bibliotecas utilizadas

| Biblioteca | Para que serve |
|---|---|
| `osmnx` | Baixar dados geográficos do OpenStreetMap |
| `geopandas` | Manipular dados geográficos como tabelas |
| `folium` | Criar mapas interativos em HTML (baseado no Leaflet.js) |
| `branca` | Injetar HTML/CSS/JavaScript customizado no mapa |
| `scipy` | Busca eficiente de vizinhos próximos (usado na análise de gaps) |

## 1. Instalação e configuração

In [1]:
!pip install osmnx geopandas folium


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import osmnx as ox
import geopandas as gpd
import pandas as pd
import folium
from branca.element import Element

# Criar pastas para organizar os arquivos gerados
for pasta in ["dados", "resultados"]:
    os.makedirs(pasta, exist_ok=True)

print("Bibliotecas importadas e pastas criadas com sucesso!")

Bibliotecas importadas e pastas criadas com sucesso!


## 2. Download dos dados do OpenStreetMap

Baixamos dois tipos de infraestrutura cicloviária:

| Tag OSM | Significado |
|---|---|
| `highway=cycleway` | Via exclusiva para bicicletas |
| `cycleway=lane/track/shared_lane/shared` | Ciclofaixa ou via compartilhada em rua normal |

> **Nota:** O download pode levar de 30s a 2 minutos.

### Mas antes: quais tags existem?

As tags acima não são as únicas que o OpenStreetMap oferece para ciclovias. Antes de decidir o que baixar, vamos ver **o quadro completo** — quantas vias existem no RJ para cada tipo de tag.

O módulo `osm_tags_reference` faz essa consulta automaticamente. Na primeira execução, ele baixa os dados do OSM e salva em cache (pasta `dados/`). Nas execuções seguintes, carrega do cache instantaneamente.

In [ ]:
# Consultar todas as tags de ciclovia disponiveis no Rio de Janeiro
# Primeira execucao: ~1-2 min (baixa do OSM). Depois: instantaneo (cache).
from osm_tags_reference import listar_tags_ciclovia

tags_rj = listar_tags_ciclovia("Rio de Janeiro, Brazil")

print("Tags de ciclovia disponiveis no Rio de Janeiro:\n")
print(tags_rj.to_string(index=False))
print(f"\nTotal: {tags_rj['contagem'].sum()} trechos de via com alguma tag de ciclovia")

### Como interpretar esta tabela

A tabela mostra três colunas:
- **tag** — o par chave=valor no padrão OSM (ex: `highway=cycleway`)
- **descrição** — o que essa tag significa em português
- **contagem** — quantas vias com essa tag existem no Rio de Janeiro

#### Por que não usamos todas as tags?

Neste notebook, usamos apenas 5 tags: `highway=cycleway`, `cycleway=lane`, `cycleway=track`, `cycleway=shared_lane` e `cycleway=shared`. As outras ficam de fora porque:

| Tag ignorada | Motivo |
|---|---|
| `bicycle=yes` | Muito ampla — inclui qualquer rua onde bicicletas são permitidas, mesmo sem infraestrutura |
| `bicycle=designated` | Similar — indica que bicicletas são o uso principal, mas nem sempre tem faixa própria |
| `cycleway=opposite/opposite_lane` | Raras no RJ e indicam contramão, não infraestrutura nova |

#### Nossa classificação

| Classificação no mapa | Tags OSM usadas | Por que? |
|---|---|---|
| **Ciclovia Exclusiva** (vermelho) | `highway=cycleway` | Via inteiramente dedicada a bicicletas |
| **Ciclofaixa Dedicada** (azul) | `cycleway=lane`, `cycleway=track` | Faixa própria na rua, pintada ou separada fisicamente |
| **Via Compartilhada** (laranja) | `cycleway=shared_lane`, `cycleway=shared` | Sem separação física — ciclista divide espaço |

> **Desafio rápido:** Tente rodar `listar_tags_ciclovia("São Paulo, Brazil")` ou `listar_tags_ciclovia("Curitiba, Brazil")`. Quais cidades tem mais infraestrutura? Quais tags aparecem lá que não aparecem no RJ?

Para saber mais sobre cada tag, consulte o [wiki do OpenStreetMap](https://wiki.openstreetmap.org/wiki/Key:cycleway).

In [3]:
cidade = "Rio de Janeiro, Brazil"

# === FONTE 1: Ciclovias exclusivas (highway=cycleway) ===
print(f"Baixando ciclovias exclusivas de '{cidade}'...")
ciclovias_exclusivas = ox.features_from_place(cidade, tags={"highway": "cycleway"})
ciclovias_exclusivas = ciclovias_exclusivas[
    ciclovias_exclusivas.geometry.geom_type == "LineString"
].copy()
ciclovias_exclusivas["tipo_ciclovia"] = "Ciclovia Exclusiva"
print(f"  Encontradas: {len(ciclovias_exclusivas)} vias")

# === FONTE 2: Ciclofaixas em ruas normais ===
print(f"\nBaixando ciclofaixas em vias normais...")
ciclofaixas = ox.features_from_place(cidade, tags={"cycleway": True})
ciclofaixas = ciclofaixas[
    ciclofaixas.geometry.geom_type == "LineString"
].copy()

# Filtrar apenas os que realmente têm infraestrutura
valores_validos = ["lane", "track", "shared_lane", "shared", "yes"]
ciclofaixas = ciclofaixas[
    ciclofaixas["cycleway"].apply(
        lambda x: str(x) if isinstance(x, list) else x
    ).isin(valores_validos)
].copy()

def classificar_ciclofaixa(valor):
    """Classifica o tipo de ciclofaixa baseado na tag do OpenStreetMap."""
    valor = str(valor)
    if valor in ["lane", "track"]:
        return "Ciclofaixa Dedicada"
    else:
        return "Via Compartilhada"

ciclofaixas["tipo_ciclovia"] = ciclofaixas["cycleway"].apply(classificar_ciclofaixa)
print(f"  Encontradas: {len(ciclofaixas)} vias")

# === COMBINAR AS DUAS FONTES ===
colunas = ["geometry", "tipo_ciclovia", "name"]
for col in colunas:
    if col not in ciclovias_exclusivas.columns:
        ciclovias_exclusivas[col] = None
    if col not in ciclofaixas.columns:
        ciclofaixas[col] = None

infraestrutura_bike = gpd.GeoDataFrame(
    pd.concat([ciclovias_exclusivas[colunas], ciclofaixas[colunas]], ignore_index=True),
    geometry="geometry",
    crs=ciclovias_exclusivas.crs,
)

print(f"\n{'='*50}")
print(f"TOTAL: {len(infraestrutura_bike)} trechos de infraestrutura cicloviária")
print(f"{'='*50}")

Baixando ciclovias exclusivas de 'Rio de Janeiro, Brazil'...


  Encontradas: 435 vias

Baixando ciclofaixas em vias normais...


  Encontradas: 191 vias

TOTAL: 626 trechos de infraestrutura cicloviária


## 3. Exploração rápida dos dados

In [ ]:
# Quantas vias de cada tipo encontramos?
print("Quantidade de vias por tipo:")
print(infraestrutura_bike["tipo_ciclovia"].value_counts())

# Exemplos de nomes de ciclovias conhecidas
nomes = infraestrutura_bike["name"].dropna().unique()
print(f"\nExemplos de vias com nome ({len(nomes)} no total):")
for nome in sorted(nomes)[:10]:
    print(f"  - {nome}")

# Primeiras linhas da tabela (cada linha = um trecho de via)
infraestrutura_bike.head(10)

### Simplificar geometrias

Antes de criar o mapa, simplificamos as geometrias para o arquivo HTML não ficar pesado demais.

A simplificação **reduz o número de pontos** em cada linha, mantendo o formato geral. O parâmetro `tolerance=0.0005` equivale a ~50 metros de precisão — suficiente para visualização, mas você não usaria isso para cálculo de distâncias exatas.

In [ ]:
# Contar pontos antes e depois para ver o efeito
pontos_antes = infraestrutura_bike.geometry.apply(lambda g: len(g.coords)).sum()

# Diminua o tolerance (ex: 0.0001) para MAIS detalhes; aumente (ex: 0.001) para MENOS detalhes e melhor performance
infraestrutura_bike["geometry"] = infraestrutura_bike["geometry"].simplify(tolerance=0.0005)

pontos_depois = infraestrutura_bike.geometry.apply(lambda g: len(g.coords)).sum()
reducao = (1 - pontos_depois / pontos_antes) * 100

print(f"Pontos antes: {pontos_antes:,}")
print(f"Pontos depois: {pontos_depois:,}")
print(f"Reducao: {reducao:.0f}% (mapa mais leve, mesma forma visual)")

## 4. Análise de Gaps — Descontinuidades na Malha Cicloviária

### O que é um gap?

Imagine que você está pedalando numa ciclovia e ela simplesmente **acaba**. Você precisa pedalar 80 metros numa avenida movimentada até a próxima ciclovia começar. Esse trecho sem infraestrutura é um **gap** (lacuna).

```
Ciclovia A ════════════╸     ╺════════════ Ciclovia B
                        └─ gap (80m) ─┘
```

**Definição técnica:** um gap é um par de endpoints (pontos finais) de ciclovias *diferentes* que estão próximos (< 100 metros) mas **não conectados**.

### Por que gaps importam?

- **Segurança:** ciclistas são forçados a entrar no tráfego de veículos
- **Usabilidade:** muitas pessoas desistem de pedalar por causa dos gaps
- **Política pública:** identificar gaps ajuda a priorizar **onde construir** o próximo trecho

### Como o algoritmo funciona

O módulo `gap_analysis.py` implementa a detecção em 5 passos:

| Passo | O que faz | Por que |
|---|---|---|
| 1. Extrair endpoints | Pega o primeiro e último ponto de cada ciclovia | São os pontos onde a ciclovia "começa" e "termina" |
| 2. Reprojetar para metros | Converte coordenadas de graus para metros (EPSG:31983) | Para medir distância em metros, não em graus |
| 3. Arvore KD | Estrutura de dados para busca eficiente de vizinhos | Sem isso, comparar todos os pares seria O(n^2) — muito lento |
| 4. Filtrar | Remove pares do mesmo segmento e endpoints já conectados | Endpoints da mesma ciclovia não são gaps; endpoints sobrepostos já estão ligados |
| 5. Resultado | Retorna os gaps como LineStrings com distância em metros | Cada gap é uma "linha imaginária" entre dois endpoints desconectados |

> **Threshold de 100 metros:** Escolhemos 100m porque quarteirões em áreas urbanas do RJ tem ~100-150m. Gaps maiores que isso provavelmente são bairros diferentes, não descontinuidades.

In [ ]:
# Importar o modulo de deteccao de gaps
from gap_analysis import detectar_gaps

# Detectar gaps com threshold de 100 metros
print("Analisando descontinuidades na malha cicloviaria...\n")
gaps_educativo = detectar_gaps(infraestrutura_bike, threshold_metros=100)

# Filtrar ruido de digitalizacao: gaps < 5m geralmente sao imprecisao
# do mapeamento no OSM, nao descontinuidades reais
gaps_educativo = gaps_educativo[gaps_educativo["distancia_m"] >= 5].copy()

# === ESTATISTICAS ===
print(f"Total de gaps encontrados: {len(gaps_educativo)}")
print(f"Distancia media:   {gaps_educativo['distancia_m'].mean():.1f} metros")
print(f"Distancia mediana:  {gaps_educativo['distancia_m'].median():.1f} metros")
print(f"Menor gap:          {gaps_educativo['distancia_m'].min():.1f} metros")
print(f"Maior gap:          {gaps_educativo['distancia_m'].max():.1f} metros")

# Distribuicao por faixa de distancia
print(f"\nDistribuicao por faixa:")
faixas = pd.cut(gaps_educativo["distancia_m"], bins=[0, 20, 50, 100],
                labels=["  0-20m (pequenos)", " 20-50m (medios)", "50-100m (grandes)"])
print(faixas.value_counts().sort_index().to_string())

# Os 10 maiores gaps (onde a malha esta mais fragmentada)
print(f"\nTop 10 maiores gaps:")
top = gaps_educativo.nlargest(10, "distancia_m")[["distancia_m", "lat_a", "lng_a", "lat_b", "lng_b"]]
for i, (_, row) in enumerate(top.iterrows(), 1):
    print(f"  {i}. {row['distancia_m']:.0f}m -- de ({row['lat_a']:.4f}, {row['lng_a']:.4f}) "
          f"ate ({row['lat_b']:.4f}, {row['lng_b']:.4f})")

### Interpretando os resultados

O que esses números dizem sobre a malha cicloviária do RJ:

- **Muitos gaps** = a rede é **fragmentada** — ciclovias existem mas não formam um sistema conectado
- **Mediana menor que a media** = existem muitos gaps pequenos (fáceis de resolver com pouco investimento) e alguns gaps muito grandes
- **Gaps 0-20m** são os mais "fáceis" de resolver — as vezes basta estender a pintura ou colocar uma rampa

Na seção 6 (Gerar o Mapa), os gaps aparecem como **linhas tracejadas rosas**. Você pode clicar na legenda (canto inferior esquerdo) para ligar/desligar a camada "Gap na Malha".

### Limitações do algoritmo atual

Você vai notar que o algoritmo detecta alguns gaps "falsos". Por exemplo, em praças com ciclovias circulares que não se conectam a malha principal, ou em trechos onde duas ciclovias passam perto mas em níveis diferentes (viaduto, túnel). Isso acontece porque o algoritmo **só olha distância entre endpoints**, sem considerar se a conexão faria sentido viário.

Possíveis melhorias (veja o Desafio 3 nas Tarefas Abertas):
- Filtrar gaps por tipo de via adjacente (ignorar gaps dentro de parques)
- Considerar a topologia da rede (o gap conectaria duas partes realmente desconectadas?)
- Usar ângulo entre os segmentos (gaps em direções opostas provavelmente não são gaps reais)

> **Desafio rápido:** Mude o threshold para 50 metros e rode a célula acima de novo. Quantos gaps sobram? O que isso diz sobre a conectividade da malha?

## 5. O Mapa Interativo: Python + JavaScript

Até agora, usamos **Python** para baixar dados, analisar gaps e criar o mapa. Mas o Folium gera um arquivo HTML *estático* — você pode ver as ciclovias, mas não pode interagir (clicar, preencher formulários, salvar dados).

Para adicionar interatividade, precisamos de **JavaScript** — a linguagem que roda dentro do navegador. Usamos a biblioteca `branca.Element` para injetar código JS diretamente no HTML que o Folium gera.

### Você precisa saber JavaScript?

**Não.** O notebook funciona sem entender JS. Mas se quiser **customizar o mapa** (mudar categorias, cores, adicionar campos), ajuda saber onde mexer. Por isso explicamos a estrutura abaixo.

### Anatomia do bloco de código

O bloco abaixo tem ~500 linhas. Parece muito, mas está organizado em seções claras:

```
ESTILOS CSS (visual)         -> Como o modal, legenda e menu aparecem na tela
HTML (estrutura)             -> Formulário de reporte, botões, legenda, menu lateral
CONFIGURAÇÃO                 -> Cores por severidade, chave do localStorage
INICIALIZAÇÃO                -> Polling até o mapa Leaflet existir no DOM
TOGGLE DE CAMADAS            -> Legenda clicável que liga/desliga camadas
MENU LATERAL                 -> Hamburguer -> painel com estatísticas e export/import
MODAL                        -> Formulário que abre ao clicar no mapa
UPLOAD DE IMAGENS            -> Compressão de fotos para caber no localStorage
SALVAR PROBLEMA              -> Cria objeto JS + marcador no mapa + persiste
MARCADORES                   -> Ícones coloridos por severidade com popup
PERSISTÊNCIA (localStorage)  -> Salvar/carregar dados do navegador
EXPORTAR / IMPORTAR          -> Download/upload de arquivo JSON
```

Cada seção está marcada com `// ====` no código — use isso para navegar.

### O que você pode ajustar facilmente

| O que mudar | Onde encontrar no código | Como |
|---|---|---|
| Categorias de problemas | `<select id="sel-categoria">` no HTML | Adicione/remova `<option>` |
| Cores de severidade | `CORES_SEVERIDADE` no JS (início) | Mude os códigos hexadecimais |
| Tamanho máximo das fotos | `comprimirImagem(ev.target.result, 800, 0.7, ...)` | `800` = pixels, `0.7` = qualidade JPEG |
| Chave de storage | `STORAGE_KEY = 'problemas_ciclovia_rj'` | Mude o nome para separar projetos |
| Texto da instrução | `<div id="instrucao-clique">` no HTML | Mude o texto entre as tags |
| Campos do formulário | Bloco `<div id="modal-form">` no HTML | Adicione novos `<label>` e `<select>` ou `<input>` |

### JavaScript vs Python — guia rápido

Se você conhece Python mas não JavaScript, está tabela ajuda a ler o código:

| Conceito JS | Equivalente Python | Exemplo no código |
|---|---|---|
| `var x = 5` | `x = 5` | `var problemas = []` |
| `function foo() { }` | `def foo():` | `function salvarProblema() { ... }` |
| `if (x === 5) { }` | `if x == 5:` | Validações no modal |
| `for (var i = 0; i < n; i++)` | `for i in range(n):` | Loop nos marcadores |
| `document.getElementById('x')` | (manipula HTML) | Pegar valor dos campos do formulário |
| `localStorage.setItem(k, v)` | `open('arq', 'w').write(v)` | Salvar problemas no navegador |
| `JSON.stringify(obj)` | `json.dumps(obj)` | Serializar para export |
| `L.marker([lat, lng])` | `folium.Marker(location=...)` | Adicionar marcador ao mapa |
| `addEventListener('click', fn)` | (reage a eventos do usuário) | Upload de fotos |

In [5]:
# ============================================================
# JavaScript e CSS para: reportar problemas + legenda interativa + menu
# ============================================================

JAVASCRIPT_PROBLEMAS = """
<style>
/* ========== MODAL DE REPORTE ========== */
#modal-overlay {
    display: none; position: fixed; top: 0; left: 0;
    width: 100%; height: 100%; background: rgba(0,0,0,0.5);
    z-index: 10000; justify-content: center; align-items: center;
}
#modal-form {
    background: white; border-radius: 12px; padding: 24px; width: 420px;
    max-height: 85vh; overflow-y: auto; box-shadow: 0 8px 32px rgba(0,0,0,0.3);
    font-family: 'Segoe UI', Arial, sans-serif;
}
#modal-form h3 { margin: 0 0 16px 0; color: #333; font-size: 18px; }
#modal-form label { display: block; margin: 12px 0 4px 0; font-weight: 600; color: #555; font-size: 13px; }
#modal-form select, #modal-form textarea {
    width: 100%; padding: 8px 10px; border: 1px solid #ccc;
    border-radius: 6px; font-size: 14px; box-sizing: border-box;
}
#modal-form textarea { height: 80px; resize: vertical; }
#img-preview { display: flex; flex-wrap: wrap; gap: 8px; margin-top: 8px; }
#img-preview img { width: 70px; height: 70px; object-fit: cover; border-radius: 6px; border: 1px solid #ddd; cursor: pointer; }
.modal-buttons { display: flex; gap: 10px; margin-top: 18px; }
.btn-salvar { flex: 1; padding: 10px; background: #2196F3; color: white; border: none; border-radius: 6px; font-size: 14px; font-weight: 600; cursor: pointer; }
.btn-salvar:hover { background: #1976D2; }
.btn-cancelar { flex: 1; padding: 10px; background: #eee; color: #333; border: none; border-radius: 6px; font-size: 14px; cursor: pointer; }
.btn-cancelar:hover { background: #ddd; }

/* ========== LEGENDA INTERATIVA ========== */
#legenda {
    position: fixed; bottom: 20px; left: 20px; z-index: 9999;
    background: white; border-radius: 10px; padding: 14px;
    box-shadow: 0 2px 12px rgba(0,0,0,0.2);
    font-family: 'Segoe UI', Arial, sans-serif; font-size: 13px;
    min-width: 190px; user-select: none;
}
#legenda h4 { margin: 0 0 10px 0; font-size: 14px; color: #333; }
.legenda-item {
    cursor: pointer; padding: 5px 4px; border-radius: 4px;
    transition: opacity 0.2s, background 0.15s; display: flex; align-items: center; gap: 8px;
}
.legenda-item:hover { background: #f0f0f0; }
.legenda-item.desligado { opacity: 0.3; }
.legenda-linha { display: inline-block; width: 22px; height: 3px; vertical-align: middle; flex-shrink: 0; }
.legenda-tracejado { display: inline-block; width: 22px; height: 0; border-top: 3px dashed #E91E63; vertical-align: middle; flex-shrink: 0; }
.legenda-sep { margin: 6px 0; border-top: 1px solid #eee; }
.legenda-label { font-size: 12px; color: #888; margin: 6px 0 4px 0; }

/* ========== BOTAO HAMBURGUER ========== */
#btn-menu {
    position: fixed; top: 12px; right: 12px; z-index: 9999;
    background: white; width: 42px; height: 42px; border-radius: 10px;
    display: flex; align-items: center; justify-content: center;
    font-size: 22px; cursor: pointer; box-shadow: 0 2px 8px rgba(0,0,0,0.2);
    color: #444; transition: background 0.15s;
}
#btn-menu:hover { background: #f0f0f0; }

/* ========== MENU LATERAL ========== */
#menu-lateral {
    position: fixed; top: 0; right: -320px; width: 300px; height: 100%;
    z-index: 10001; background: white; box-shadow: -4px 0 20px rgba(0,0,0,0.15);
    transition: right 0.3s ease; padding: 0; overflow-y: auto;
    font-family: 'Segoe UI', Arial, sans-serif;
}
#menu-lateral.aberto { right: 0; }
#menu-overlay {
    display: none; position: fixed; top: 0; left: 0; width: 100%; height: 100%;
    background: rgba(0,0,0,0.3); z-index: 10000;
}
#menu-overlay.aberto { display: block; }
.menu-header {
    display: flex; justify-content: space-between; align-items: center;
    padding: 18px 20px 14px; border-bottom: 1px solid #eee;
}
.menu-header h4 { margin: 0; font-size: 16px; color: #333; }
.menu-close { font-size: 22px; cursor: pointer; color: #999; padding: 4px 8px; border-radius: 4px; }
.menu-close:hover { background: #f0f0f0; color: #333; }
.menu-stats { padding: 16px 20px; background: #f8f9fa; border-bottom: 1px solid #eee; }
.menu-stat { display: flex; justify-content: space-between; padding: 4px 0; font-size: 13px; color: #555; }
.menu-stat .num { font-weight: 600; color: #333; }
.menu-section { padding: 14px 20px; border-bottom: 1px solid #f0f0f0; }
.menu-section h5 { margin: 0 0 10px 0; font-size: 13px; color: #999; text-transform: uppercase; letter-spacing: 0.5px; }
.menu-section p { margin: 0; font-size: 13px; color: #666; line-height: 1.5; }
.menu-section button {
    display: block; width: 100%; padding: 10px 14px; margin: 6px 0;
    border: 1px solid #e0e0e0; border-radius: 8px; background: white;
    font-size: 13px; cursor: pointer; text-align: left;
    transition: background 0.15s; color: #333;
}
.menu-section button:hover { background: #f5f5f5; }
.menu-section .btn-danger { color: #c62828; border-color: #e57373; }
.menu-section .btn-danger:hover { background: #ffebee; }

/* ========== LIGHTBOX ========== */
#lightbox {
    display: none; position: fixed; top: 0; left: 0; width: 100%; height: 100%;
    background: rgba(0,0,0,0.85); z-index: 20000;
    justify-content: center; align-items: center; cursor: pointer;
}
#lightbox img { max-width: 90%; max-height: 90%; border-radius: 8px; }
</style>

<!-- Modal de reporte -->
<div id=\"modal-overlay\">
  <div id=\"modal-form\">
    <h3>Reportar Problema</h3>
    <p id=\"modal-coords\" style=\"color:#888; font-size:12px; margin:0 0 8px 0;\"></p>
    <label for=\"sel-categoria\">Categoria</label>
    <select id=\"sel-categoria\">
      <option value=\"Buraco/Desnivel\">Buraco / Desnivel</option>
      <option value=\"Falta de Sinalizacao\">Falta de Sinalizacao</option>
      <option value=\"Obstrucao\">Obstrucao</option>
      <option value=\"Trecho Perigoso\">Trecho Perigoso</option>
      <option value=\"Iluminacao\">Iluminacao</option>
      <option value=\"Outro\">Outro</option>
    </select>
    <label for=\"txt-descricao\">Descricao</label>
    <textarea id=\"txt-descricao\" placeholder=\"Descreva o problema...\"></textarea>
    <label for=\"sel-severidade\">Severidade</label>
    <select id=\"sel-severidade\">
      <option value=\"Info\">Info (sugestao/observacao)</option>
      <option value=\"Baixo\">Baixo (problema menor)</option>
      <option value=\"Medio\" selected>Medio (requer atencao)</option>
      <option value=\"Critico\">Critico (risco de acidente)</option>
    </select>
    <label for=\"input-fotos\">Fotos (opcional, multiplas)</label>
    <input type=\"file\" id=\"input-fotos\" accept=\"image/*\" multiple style=\"font-size:13px; margin-top:4px;\">
    <div id=\"img-preview\"></div>
    <div class=\"modal-buttons\">
      <button class=\"btn-salvar\" onclick=\"salvarProblema()\">Salvar</button>
      <button class=\"btn-cancelar\" onclick=\"fecharModal()\">Cancelar</button>
    </div>
  </div>
</div>

<!-- Legenda interativa -->
<div id=\"legenda\">
  <h4>Legenda</h4>
  <div class=\"legenda-item\" data-camada=\"Ciclovia Exclusiva\" onclick=\"toggleCamada(this)\">
    <span class=\"legenda-linha\" style=\"background:red;\"></span> Ciclovia Exclusiva
  </div>
  <div class=\"legenda-item\" data-camada=\"Ciclofaixa Dedicada\" onclick=\"toggleCamada(this)\">
    <span class=\"legenda-linha\" style=\"background:blue;\"></span> Ciclofaixa Dedicada
  </div>
  <div class=\"legenda-item\" data-camada=\"Via Compartilhada\" onclick=\"toggleCamada(this)\">
    <span class=\"legenda-linha\" style=\"background:orange;\"></span> Via Compartilhada
  </div>
  <div class=\"legenda-item\" data-camada=\"Gaps na Malha\" onclick=\"toggleCamada(this)\">
    <span class=\"legenda-tracejado\"></span> Gap na Malha
  </div>
  <div class=\"legenda-sep\"></div>
  <div class=\"legenda-label\">Problemas reportados:</div>
  <div style=\"display:flex; gap:6px; flex-wrap:wrap; font-size:12px;\">
    <span style=\"color:#2196F3;\">&#9679; Info</span>
    <span style=\"color:#4CAF50;\">&#9679; Baixo</span>
    <span style=\"color:#FF9800;\">&#9679; Medio</span>
    <span style=\"color:#F44336;\">&#9679; Critico</span>
  </div>
</div>

<!-- Botao hamburguer -->
<div id=\"btn-menu\" onclick=\"toggleMenu()\">&#9776;</div>

<!-- Overlay do menu -->
<div id=\"menu-overlay\" onclick=\"toggleMenu()\"></div>

<!-- Menu lateral -->
<div id=\"menu-lateral\">
  <div class=\"menu-header\">
    <h4>Ferramentas</h4>
    <span class=\"menu-close\" onclick=\"toggleMenu()\">&times;</span>
  </div>
  <div class=\"menu-section\">
    <h5>Como usar</h5>
    <p>Clique em qualquer ponto do mapa para reportar um problema. Preencha a categoria, descricao e severidade. Anexe fotos se quiser.</p>
  </div>
  <div class=\"menu-stats\">
    <div class=\"menu-stat\"><span>Problemas reportados</span><span class=\"num\" id=\"stat-problemas\">0</span></div>
    <div class=\"menu-stat\"><span>Gaps detectados</span><span class=\"num\" id=\"stat-gaps\">0</span></div>
    <div class=\"menu-stat\"><span>Trechos de ciclovia</span><span class=\"num\" id=\"stat-ciclovias\">0</span></div>
  </div>
  <div class=\"menu-section\">
    <h5>Dados</h5>
    <button onclick=\"exportarJSON()\">Exportar JSON</button>
    <button onclick=\"document.getElementById('input-importar').click()\">Importar JSON</button>
    <input type=\"file\" id=\"input-importar\" accept=\".json\" style=\"display:none\" onchange=\"importarJSON(event)\">
  </div>
  <div class=\"menu-section\">
    <button class=\"btn-danger\" onclick=\"limparTodos()\">Limpar Tudo</button>
  </div>
</div>

<!-- Lightbox -->
<div id=\"lightbox\" onclick=\"this.style.display='none'\">
  <img id=\"lightbox-img\" src=\"\">
</div>

<script>
// ============================================================
//  CONFIGURACAO
// ============================================================
var STORAGE_KEY = 'problemas_ciclovia_rj';
var CORES_SEVERIDADE = {
    'Info': '#2196F3', 'Baixo': '#4CAF50',
    'Medio': '#FF9800', 'Critico': '#F44336'
};

var problemas = [];
var marcadores = [];
var clickLat = 0, clickLng = 0;
var imagensBase64 = [];
var proximoId = 1;
var mapaLeaflet = null;
var camadasMapa = {};

// ============================================================
//  INICIALIZACAO (polling)
// ============================================================
function inicializarMapa() {
    for (var nome in window) {
        if (nome.indexOf('map_') === 0 && window[nome] instanceof L.Map) {
            mapaLeaflet = window[nome];
            break;
        }
    }
    if (!mapaLeaflet) { setTimeout(inicializarMapa, 200); return; }

    // Resolver nomes das camadas para referencias reais
    // CAMADAS_NOMES e injetado pelo Python com os nomes JS das variaveis
    if (typeof CAMADAS_NOMES !== 'undefined') {
        for (var nome in CAMADAS_NOMES) {
            var jsVar = CAMADAS_NOMES[nome];
            if (window[jsVar]) {
                camadasMapa[nome] = window[jsVar];
            }
        }
    }

    mapaLeaflet.on('click', function(e) {
        clickLat = e.latlng.lat;
        clickLng = e.latlng.lng;
        abrirModal(clickLat, clickLng);
    });

    carregarDoStorage();
    atualizarEstatisticas();
}
inicializarMapa();

// ============================================================
//  TOGGLE DE CAMADAS
// ============================================================
function toggleCamada(el) {
    var nome = el.getAttribute('data-camada');
    var camada = camadasMapa[nome];
    if (!camada) { console.warn('Camada nao encontrada:', nome); return; }

    if (mapaLeaflet.hasLayer(camada)) {
        mapaLeaflet.removeLayer(camada);
        el.classList.add('desligado');
    } else {
        mapaLeaflet.addLayer(camada);
        el.classList.remove('desligado');
    }
}

// ============================================================
//  MENU LATERAL
// ============================================================
function toggleMenu() {
    document.getElementById('menu-lateral').classList.toggle('aberto');
    document.getElementById('menu-overlay').classList.toggle('aberto');
}

function atualizarEstatisticas() {
    document.getElementById('stat-problemas').textContent = problemas.length;
    if (typeof TOTAL_GAPS !== 'undefined')
        document.getElementById('stat-gaps').textContent = TOTAL_GAPS;
    if (typeof TOTAL_CICLOVIAS !== 'undefined')
        document.getElementById('stat-ciclovias').textContent = TOTAL_CICLOVIAS;
}

// ============================================================
//  MODAL
// ============================================================
function abrirModal(lat, lng) {
    document.getElementById('modal-coords').textContent =
        'Lat: ' + lat.toFixed(6) + ', Lng: ' + lng.toFixed(6);
    document.getElementById('sel-categoria').selectedIndex = 0;
    document.getElementById('txt-descricao').value = '';
    document.getElementById('sel-severidade').value = 'Medio';
    document.getElementById('input-fotos').value = '';
    document.getElementById('img-preview').innerHTML = '';
    imagensBase64 = [];
    document.getElementById('modal-overlay').style.display = 'flex';
}

function fecharModal() {
    document.getElementById('modal-overlay').style.display = 'none';
    imagensBase64 = [];
}

// ============================================================
//  UPLOAD DE IMAGENS
// ============================================================
document.getElementById('input-fotos').addEventListener('change', function(e) {
    var arquivos = e.target.files;
    var preview = document.getElementById('img-preview');
    preview.innerHTML = ''; imagensBase64 = [];
    for (var i = 0; i < arquivos.length; i++) {
        (function(arquivo) {
            var reader = new FileReader();
            reader.onload = function(ev) {
                comprimirImagem(ev.target.result, 800, 0.7, function(comprimida) {
                    imagensBase64.push(comprimida);
                    var img = document.createElement('img');
                    img.src = comprimida; img.title = arquivo.name;
                    img.onclick = function() { abrirLightbox(comprimida); };
                    preview.appendChild(img);
                });
            };
            reader.readAsDataURL(arquivo);
        })(arquivos[i]);
    }
});

function comprimirImagem(dataUrl, maxLado, qualidade, callback) {
    var img = new Image();
    img.onload = function() {
        var w = img.width, h = img.height;
        if (w > maxLado || h > maxLado) {
            if (w > h) { h = Math.round(h * maxLado / w); w = maxLado; }
            else { w = Math.round(w * maxLado / h); h = maxLado; }
        }
        var c = document.createElement('canvas'); c.width = w; c.height = h;
        c.getContext('2d').drawImage(img, 0, 0, w, h);
        callback(c.toDataURL('image/jpeg', qualidade));
    };
    img.src = dataUrl;
}

// ============================================================
//  SALVAR PROBLEMA
// ============================================================
function salvarProblema() {
    var descricao = document.getElementById('txt-descricao').value.trim();
    if (!descricao) { alert('Por favor, descreva o problema.'); return; }
    var problema = {
        id: proximoId++, lat: clickLat, lng: clickLng,
        categoria: document.getElementById('sel-categoria').value,
        descricao: descricao,
        severidade: document.getElementById('sel-severidade').value,
        imagens: imagensBase64.slice(),
        data_criacao: new Date().toISOString()
    };
    problemas.push(problema);
    adicionarMarcador(problema);
    salvarNoStorage();
    atualizarEstatisticas();
    fecharModal();
}

// ============================================================
//  MARCADORES
// ============================================================
function adicionarMarcador(problema) {
    var cor = CORES_SEVERIDADE[problema.severidade] || '#999';
    var icone = L.divIcon({
        className: '',
        html: '<div style=\"background:' + cor + '; width:22px; height:22px; border-radius:50%; border:3px solid white; box-shadow:0 2px 6px rgba(0,0,0,0.4);\"></div>',
        iconSize: [22, 22], iconAnchor: [11, 11]
    });
    var h = '<div style=\"font-family:Segoe UI,Arial; min-width:200px; max-width:300px;\">';
    h += '<b style=\"font-size:14px;\">' + problema.categoria + '</b>';
    h += '<span style=\"float:right; padding:2px 8px; border-radius:4px; font-size:11px; color:white; background:' + cor + ';\">' + problema.severidade + '</span>';
    h += '<p style=\"margin:8px 0; font-size:13px; color:#555;\">' + problema.descricao + '</p>';
    if (problema.imagens && problema.imagens.length > 0) {
        h += '<div style=\"display:flex; flex-wrap:wrap; gap:4px; margin:8px 0;\">';
        for (var i = 0; i < problema.imagens.length; i++)
            h += '<img src=\"' + problema.imagens[i] + '\" style=\"width:60px; height:60px; object-fit:cover; border-radius:4px; cursor:pointer;\" onclick=\"abrirLightbox(this.src)\">';
        h += '</div>';
    }
    var d = new Date(problema.data_criacao);
    h += '<div style=\"font-size:11px; color:#999; margin-top:6px;\">' + d.toLocaleDateString('pt-BR') + ' ' + d.toLocaleTimeString('pt-BR', {hour:'2-digit', minute:'2-digit'}) + '</div>';
    h += '<button onclick=\"removerProblema(' + problema.id + ')\" style=\"margin-top:8px; padding:4px 12px; background:#f44336; color:white; border:none; border-radius:4px; cursor:pointer; font-size:12px;\">Remover</button>';
    h += '</div>';
    var m = L.marker([problema.lat, problema.lng], {icon: icone}).bindPopup(h, {maxWidth: 320}).addTo(mapaLeaflet);
    m._problemaId = problema.id;
    marcadores.push(m);
}

function removerProblema(id) {
    problemas = problemas.filter(function(p) { return p.id !== id; });
    for (var i = marcadores.length - 1; i >= 0; i--) {
        if (marcadores[i]._problemaId === id) {
            mapaLeaflet.removeLayer(marcadores[i]); marcadores.splice(i, 1); break;
        }
    }
    salvarNoStorage(); atualizarEstatisticas();
}

// ============================================================
//  PERSISTENCIA
// ============================================================
function salvarNoStorage() {
    try { localStorage.setItem(STORAGE_KEY, JSON.stringify(problemas)); }
    catch(e) { alert('Armazenamento cheio. Exporte os dados como JSON.'); }
}
function carregarDoStorage() {
    var dados = localStorage.getItem(STORAGE_KEY);
    if (!dados) return;
    try {
        problemas = JSON.parse(dados); proximoId = 1;
        for (var i = 0; i < problemas.length; i++) {
            if (problemas[i].id >= proximoId) proximoId = problemas[i].id + 1;
            adicionarMarcador(problemas[i]);
        }
    } catch(e) { console.error('Erro:', e); }
}

// ============================================================
//  EXPORTAR / IMPORTAR
// ============================================================
function exportarJSON() {
    if (problemas.length === 0) { alert('Nenhum problema para exportar.'); return; }
    var json = JSON.stringify({ versao: '1.0', exportado_em: new Date().toISOString(), total: problemas.length, problemas: problemas }, null, 2);
    var blob = new Blob([json], {type: 'application/json'});
    var link = document.createElement('a');
    link.href = URL.createObjectURL(blob);
    link.download = 'problemas_ciclovia_' + new Date().toISOString().slice(0,10) + '.json';
    link.click(); URL.revokeObjectURL(link.href);
    toggleMenu();
}
function importarJSON(event) {
    var arquivo = event.target.files[0]; if (!arquivo) return;
    var reader = new FileReader();
    reader.onload = function(e) {
        try {
            var dados = JSON.parse(e.target.result);
            var lista = dados.problemas || dados;
            if (!Array.isArray(lista)) { alert('Formato invalido.'); return; }
            var n = 0;
            for (var i = 0; i < lista.length; i++) {
                var p = lista[i];
                if (p.lat && p.lng && p.categoria && p.descricao) {
                    p.id = proximoId++; problemas.push(p); adicionarMarcador(p); n++;
                }
            }
            salvarNoStorage(); atualizarEstatisticas();
            alert(n + ' problema(s) importado(s)!');
        } catch(err) { alert('Erro: ' + err.message); }
    };
    reader.readAsText(arquivo); event.target.value = '';
}

// ============================================================
//  UTILIDADES
// ============================================================
function limparTodos() {
    if (problemas.length === 0) { alert('Nenhum problema para limpar.'); return; }
    if (!confirm('Remover todos os ' + problemas.length + ' problemas?')) return;
    for (var i = 0; i < marcadores.length; i++) mapaLeaflet.removeLayer(marcadores[i]);
    marcadores = []; problemas = []; proximoId = 1;
    salvarNoStorage(); atualizarEstatisticas(); toggleMenu();
}
function abrirLightbox(src) {
    document.getElementById('lightbox-img').src = src;
    document.getElementById('lightbox').style.display = 'flex';
}
</script>
"""

print(f"JavaScript definido: {len(JAVASCRIPT_PROBLEMAS)} caracteres")
print("Pronto para injetar no mapa!")

JavaScript definido: 20728 caracteres
Pronto para injetar no mapa!


> **Aviso importante: localStorage**
>
> O código acima usa `localStorage` para salvar os problemas reportados. Entenda as limitações:
>
> 1. **Não é um banco de dados.** Se você limpar o cache do navegador, os dados somem. Sempre exporte como JSON.
> 2. **Limite de espaço.** O localStorage tem ~5-10 MB por site. Com fotos comprimidas, cabe ~50-100 problemas.
> 3. **Só funciona no HTML salvo.** Dentro do Jupyter/Colab, o iframe bloqueia o acesso ao localStorage.
> 4. **Isolado por navegador.** Chrome e Firefox tem dados separados. Compartilhem via "Exportar JSON".
>
> Quer resolver essas limitações? Veja o **Desafio 4** na seção de Tarefas Abertas.

### Features do mapa interativo

O mapa que será gerado na próxima seção tem várias funcionalidades além do reporte de problemas:

| Feature | Onde fica | O que faz |
|---|---|---|
| **Legenda clicável** | Canto inferior esquerdo | Clique num item para ligar/desligar a camada no mapa |
| **Menu hamburguer** | Canto superior direito (ícone ☰) | Abre painel lateral com estatísticas e botões de export/import |
| **Instrução flutuante** | Topo central | Lembra o usuário de clicar no mapa para reportar |
| **Lightbox** | Clique numa foto no popup | Abre a imagem em tela cheia |
| **Marcadores coloridos** | Sobre o mapa | Cor indica severidade (azul=info, verde=baixo, laranja=médio, vermelho=crítico) |

## 6. Gerar o mapa final

A célula abaixo junta **tudo** que construímos até aqui:

1. **Mapa base** — CartoDB positron (fundo claro), centrado no RJ, zoom 12
2. **Camadas de ciclovias** — cada tipo (Exclusiva, Dedicada, Compartilhada) como camada separada, para poder ligar/desligar na legenda clicável
3. **Gaps** — recalcula os gaps e visualiza como linhas tracejadas rosas
4. **Registro de camadas** — injeta variáveis JS para que a legenda consiga encontrar cada camada
5. **Estatisticas** — injeta contadores para o menu lateral mostrar os números
6. **JavaScript** — injeta todo o bloco da seção 5 (modal, legenda, menu, persistência)
7. **Salvar HTML** — arquivo standalone em `resultados/mapa_problemas_ciclovias.html`

In [6]:
# Importar modulo de analise de gaps
from gap_analysis import detectar_gaps

# Cores das camadas de ciclovias
cores_por_tipo = {
    "Ciclovia Exclusiva":   "red",
    "Ciclofaixa Dedicada":  "blue",
    "Via Compartilhada":    "orange",
}

# Criar mapa base centrado no Rio de Janeiro
mapa_rj = folium.Map(
    location=[-22.9068, -43.1729],
    zoom_start=12,
    tiles="CartoDB positron",
)

# Adicionar camadas de ciclovias (cada uma num FeatureGroup nomeado)
registros_camadas = []

for tipo, cor in cores_por_tipo.items():
    subset = infraestrutura_bike[infraestrutura_bike["tipo_ciclovia"] == tipo]
    if subset.empty:
        continue

    grupo = folium.FeatureGroup(name=tipo)
    folium.GeoJson(
        subset,
        style_function=lambda x, c=cor: {"color": c, "weight": 2.5},
        tooltip=folium.GeoJsonTooltip(
            fields=["name", "tipo_ciclovia"],
            aliases=["Nome da via:", "Tipo:"],
        ),
    ).add_to(grupo)
    grupo.add_to(mapa_rj)
    registros_camadas.append((tipo, grupo.get_name()))
    print(f"  Camada '{tipo}': {len(subset)} trechos (cor: {cor})")

# --- CAMADA DE GAPS ---
print("\nAnalisando gaps na malha cicloviaria...")
gaps = detectar_gaps(infraestrutura_bike, threshold_metros=100)
gaps = gaps[gaps["distancia_m"] >= 5]
print(f"  Gaps encontrados: {len(gaps)} (>= 5m)")

grupo_gaps = folium.FeatureGroup(name="Gaps na Malha")
for _, gap in gaps.iterrows():
    coords = list(gap.geometry.coords)
    pontos = [(c[1], c[0]) for c in coords]
    folium.PolyLine(
        pontos, color="#E91E63", weight=3, dash_array="8 6",
        opacity=0.8, popup=f"Gap: {gap['distancia_m']}m",
    ).add_to(grupo_gaps)
grupo_gaps.add_to(mapa_rj)
registros_camadas.append(("Gaps na Malha", grupo_gaps.get_name()))

# Controle de camadas feito pela legenda customizada no JS

# --- INJETAR NOMES DAS VARIAVEIS JS DAS CAMADAS ---
# IMPORTANTE: as variaveis feature_group_* so existem DEPOIS dos scripts do Folium.
# Entao injetamos os NOMES como strings, e o JS resolve depois (no inicializarMapa).
linhas = [f'var CAMADAS_NOMES = {{}};']
for nome, js_var in registros_camadas:
    linhas.append(f'CAMADAS_NOMES["{nome}"] = "{js_var}";')
linhas.append(f"var TOTAL_GAPS = {len(gaps)};")
linhas.append(f"var TOTAL_CICLOVIAS = {len(infraestrutura_bike)};")

mapa_rj.get_root().html.add_child(Element(
    "<script>\n" + "\n".join(linhas) + "\n</script>"
))

# Injetar JS de problemas + legenda + menu
mapa_rj.get_root().html.add_child(Element(JAVASCRIPT_PROBLEMAS))

arquivo_mapa = os.path.join("resultados", "mapa_problemas_ciclovias.html")
mapa_rj.save(arquivo_mapa)
print(f"\nMapa salvo em: {arquivo_mapa}")

  Camada 'Ciclovia Exclusiva': 435 trechos (cor: red)
  Camada 'Ciclofaixa Dedicada': 33 trechos (cor: blue)
  Camada 'Via Compartilhada': 158 trechos (cor: orange)

Analisando gaps na malha cicloviaria...
  Gaps encontrados: 1844 (>= 5m)



Mapa salvo em: resultados\mapa_problemas_ciclovias.html


## 7. Como usar o mapa

### Instruções para os estagiários

1. **Abra o arquivo** `resultados/mapa_problemas_ciclovias.html` no navegador (Chrome, Firefox, Edge)
2. **Explore as camadas** — clique nos itens da **legenda** (canto inferior esquerdo) para ligar/desligar ciclovias e gaps
3. **Clique no mapa** no local onde há um problema
4. **Preencha o formulário:** categoria, descrição, severidade, fotos
5. **Clique em Salvar** — o marcador aparece no mapa com cor da severidade
6. **Veja estatísticas** — clique no **menu ☰** (canto superior direito) para ver contagem de problemas, gaps e ciclovias

### Exportar e compartilhar dados

- Abra o **menu ☰** e clique em **"Exportar JSON"**
- O arquivo `problemas_ciclovia_YYYY-MM-DD.json` será baixado
- Envie esse arquivo para o professor ou colegas
- Para carregar dados de outra pessoa, use **"Importar JSON"** no mesmo menu

### Dicas

- Clique num **marcador** existente para ver detalhes, fotos e botão de remover
- Clique numa **foto** no popup para abrir em tela cheia (lightbox)
- Os dados persistem no navegador entre sessões (localStorage)
- Use **"Limpar Tudo"** no menu para recomeçar do zero

### Limitações

- Fotos comprimidas para 800px e JPEG 70% (para caber no localStorage)
- localStorage tem limite de ~5-10 MB por site (~50-100 problemas com fotos)
- Cada navegador tem seu próprio storage — dados não são compartilhados entre browsers
- Se o armazenamento encher, exporte como JSON e use "Limpar Tudo"

In [7]:
# Visualizar o mapa inline (funciona no Jupyter/Colab)
# Nota: a funcionalidade de reporte so funciona 100% no HTML salvo,
# pois o iframe do Jupyter limita acesso ao localStorage.
mapa_rj

## Tarefas Abertas para Estagiários

Este notebook é o **ponto de partida**. Abaixo estão desafios para você expandir o projeto. Cada desafio tem nível de dificuldade e dicas para começar.

**Legenda:** Básico = pode fazer com o que já sabe | Intermediário = precisa pesquisar | Avançado = projeto de vários dias

---

### Desafio 1: Integrar Estações de Bike Compartilhada (Intermediário)

**Objetivo:** Adicionar uma camada no mapa com a localização das estações Bike Itau / Tembici.

**Dicas:** Pesquise se existe API pública (padrão GBFS), dados no [Data.Rio](https://www.data.rio/), ou colete manualmente. Veja o starter code na célula abaixo.

**Entrega:** Nova camada no mapa com ícone de bicicleta em cada estação.

---

### Desafio 2: Ranking de Bairros por Cobertura Cicloviária (Intermediário)

**Objetivo:** Calcular km de ciclovia por bairro e criar ranking.

**Dicas:** Dados de bairros no [Data.Rio](https://www.data.rio/) ou IBGE. Use `geopandas.sjoin()` para spatial join. Calcule comprimento com `.length` em CRS métrico.

**Entrega:** Tabela com ranking + mapa coroplético (bairros coloridos por cobertura).

---

### Desafio 3: Melhorar e Classificar Gaps (Básico a Intermediário)

**Objetivo:** O algoritmo atual detecta muitos gaps "falsos" (praças, viadutos, ciclovias isoladas). Melhore a detecção para encontrar apenas faltas de conexão significativas, e classifique os gaps restantes por gravidade.

**Dicas para melhorar a detecção:** Filtrar gaps dentro de parques/praças (tag `leisure=park`). Ignorar ciclovias que não conectam a nada. Considerar ângulo entre segmentos.

**Dicas para classificar:** Use tag `highway` do OSM para saber o tipo de via adjacente. Baixe com `ox.features_from_place(cidade, tags={"highway": True})`. Classifique: vermelho = gap em arterial, amarelo = coletora, verde = local.

**Entrega:** Algoritmo melhorado com menos falsos positivos + gaps coloridos por gravidade no mapa.

---

### Desafio 4: Persistência com Backend (Avançado)

**Objetivo:** Substituir localStorage por um backend para dados ficarem num servidor.

**Dicas:** Opção simples: [Supabase](https://supabase.com) (banco gratuito com API). Opção Python: FastAPI + SQLite (~50 linhas). No JS, troque `localStorage.setItem()` por `fetch()`.

**Entrega:** Problemas salvos num banco de dados, acessíveis de qualquer navegador.

---

### Desafio 5: Algoritmo de Rotas Seguras (Avançado)

**Objetivo:** Dado ponto A e ponto B, encontrar a rota que maximiza uso de ciclovias.

**Dicas:** `osmnx.graph_from_place()` para grafo de ruas. Pesos customizados: vias com ciclovia = peso baixo, sem = peso alto. Algoritmo: Dijkstra ou A*.

**Entrega:** Função que recebe dois pontos e desenha a rota no mapa.

---

### Desafio 6: Melhorar a Interface do Mapa (Básico)

**Objetivo:** O mapa já funciona, mas pode ser melhorado. Sugestões:

- Adicionar novas categorias de problemas (ex: "Falta de bicicletário", "Drenagem ruim")
- Mudar as cores de severidade para um esquema diferente
- Adicionar um campo "urgência" ou "data prevista" ao formulário
- Criar um modo de "marcar ponto positivo" (elogio a infraestrutura)

**Dicas:** Edite o HTML dentro da variável `JAVASCRIPT_PROBLEMAS` no notebook. Veja a tabela "O que você pode ajustar" na seção 5.

**Entrega:** Mapa com sua customização funcionando.

---

### Desafio 7: Comparar com Cidades Metropolitanas (Intermediário)

**Objetivo:** Repetir a análise para Niterói, São Gonçalo, ou outra cidade da região metropolitana.

**Dicas:** Basta mudar a variável `cidade` no início e rodar tudo de novo. Compare: número de ciclovias, número de gaps, cobertura total.

**Entrega:** Comparação entre RJ e outra cidade (tabela ou mapa lado a lado).

In [ ]:
# === DESAFIO 1: Starter code -- Estacoes de Bike Compartilhada ===
#
# Descomente e complete o codigo abaixo.
# Voce precisa descobrir onde ficam as estacoes (API, dados abertos, ou manualmente).
#
# Exemplo com coordenadas FICTICIAS (substitua por dados reais!):

# from shapely.geometry import Point
#
# estacoes = gpd.GeoDataFrame({
#     "nome": ["Estacao Botafogo", "Estacao Copacabana", "Estacao Ipanema"],
#     "capacidade": [20, 15, 12],
#     "geometry": [
#         Point(-43.1822, -22.9519),
#         Point(-43.1781, -22.9671),
#         Point(-43.2005, -22.9836),
#     ],
# }, crs="EPSG:4326")
#
# # Adicionar cada estacao ao mapa como marcador com icone de bicicleta
# for _, est in estacoes.iterrows():
#     folium.Marker(
#         location=[est.geometry.y, est.geometry.x],  # (lat, lng)
#         popup=f"{est['nome']} ({est['capacidade']} vagas)",
#         icon=folium.Icon(color="darkblue", icon="bicycle", prefix="fa"),
#     ).add_to(mapa_rj)
#
# print(f"Adicionadas {len(estacoes)} estacoes ao mapa!")

### Desafio 8: Encontrar Pontos Críticos Faltantes na Rede (Avançado)

**Objetivo:** Dado o grafo de ruas do RJ, identificar **onde a falta de ciclovias mais prejudica** a conectividade entre bairros. Este é o desafio mais ambicioso — combina teoria de grafos, dados geoespaciais, e potencialmente machine learning.

**Ideia central:** Modelar a cidade como um grafo de ruas, tentar conectar pontos entre bairros priorizando ciclovias, e identificar os trechos onde a ausência de ciclovia mais atrapalha.

Existem várias abordagens possíveis. Escolha a que mais te interessa:

<details>
<summary><b>Abordagem A: Pontos centrais de bairros (Intermediário)</b></summary>

**Ideia:** Calcular o centroide (ponto central) de cada bairro e tentar conectar bairros vizinhos por ciclovias.

**Passos:**
1. Baixar limites de bairros do [Data.Rio](https://www.data.rio/) ou do OpenStreetMap (`admin_level=9`)
2. Calcular o centroide de cada bairro com `gdf.centroid`
3. Para cada par de bairros vizinhos, calcular a rota ótima priorizando ciclovias (peso menor em arestas com ciclovia)
4. Agregar: quais trechos SEM ciclovia aparecem em MUITAS rotas? Esses são os **pontos críticos**
5. Visualizar no mapa com um heatmap de frequência

**Conceitos:** spatial join, centroide, roteamento com pesos, Dijkstra

</details>

<details>
<summary><b>Abordagem B: Pontos aleatórios + Monte Carlo (Intermediário)</b></summary>

**Ideia:** Em vez de usar centroides, gerar pontos aleatórios pela cidade e calcular rotas entre eles. A vantagem e que não depende de limites de bairros e captura padrões emergentes.

**Passos:**
1. Gerar N pontos aleatórios dentro dos limites da cidade (ex: 100-500 pontos)
2. Para cada par de pontos (ou amostra de pares), calcular rota priorizando ciclovias
3. Contar: quantas vezes cada trecho SEM ciclovia aparece nas rotas?
4. Trechos mais frequentes = gargalos da rede = pontos críticos
5. Visualizar como heatmap ou ranking de segmentos

**Conceitos:** amostragem aleatória, simulação Monte Carlo, heatmap de frequência

</details>

<details>
<summary><b>Abordagem C: Velocidades diferenciadas por tipo de via (Intermediário)</b></summary>

**Ideia:** Modelar a velocidade do ciclista de acordo com o tipo de via. Ciclovias são rápidas e seguras; avenidas são lentas e perigosas. Rotas ótimas por TEMPO naturalmente priorizam ciclovias.

**Passos:**
1. Atribuir velocidades por tipo de via:
   - Ciclovia exclusiva: 20 km/h
   - Ciclofaixa dedicada: 15 km/h
   - Via compartilhada: 12 km/h
   - Rua residencial (sem ciclovia): 10 km/h
   - Via coletora (sem ciclovia): 7 km/h
   - Via arterial (sem ciclovia): 5 km/h (perigo, semáforos, tráfego)
2. Peso de cada aresta = distância / velocidade (= tempo de percurso)
3. Comparar: rota mais CURTA vs rota mais RAPIDA
4. Onde as rotas divergem = onde faltam ciclovias que fariam diferença

**Conceitos:** grafo ponderado, Dijkstra, análise comparativa, modelagem de velocidade

</details>

<details>
<summary><b>Abordagem D: Classificador por rua com Machine Learning (Avançado)</b></summary>

**Ideia:** Treinar um modelo que aprende quais características de uma rua indicam que ela "deveria" ter ciclovia. Ruas que o modelo classifica como "deveria ter" mas não tem = gaps prioritários.

**Passos:**
1. Coletar features de cada rua via OSM: tipo (`highway` tag), largura, velocidade máxima, número de faixas, presença de ciclovia (variável alvo)
2. Treinar classificador (Random Forest, Logistic Regression) com scikit-learn
3. Prever: para cada rua SEM ciclovia, qual a probabilidade de que "deveria ter"?
4. Ranking: ruas com maior probabilidade = gaps prioritários
5. Alternativa: clustering de ruas por características + encontrar anomalias

**Conceitos:** classificação supervisionada, feature engineering, scikit-learn, validação cruzada

**Bibliotecas extras:** `scikit-learn`, `matplotlib` (para análise exploratória)

</details>

> **Dica geral:** Comece pela Abordagem A ou C — são mais acessíveis e dão resultados visuais rápidos. A Abordagem D é um projeto de pesquisa completo.

In [ ]:
# === DESAFIO 8: Starter code — Roteamento por grafos ===
# Este codigo mostra como baixar o grafo de ruas do RJ e calcular
# uma rota basica entre dois pontos. E o ponto de partida para
# todas as abordagens acima.
#
# ATENCAO: baixar o grafo do RJ inteiro pode demorar varios minutos
# e usar bastante memoria. Comece com um bairro so para testar.

# import osmnx as ox
# import networkx as nx
#
# # Baixar grafo de ruas (modo bicicleta) — comece com um bairro!
# G = ox.graph_from_place("Botafogo, Rio de Janeiro, Brazil", network_type="bike")
# print(f"Grafo: {len(G.nodes)} nos, {len(G.edges)} arestas")
#
# # Encontrar no mais proximo de dois pontos
# orig = ox.nearest_nodes(G, X=-43.1822, Y=-22.9519)  # Botafogo
# dest = ox.nearest_nodes(G, X=-43.1781, Y=-22.9671)  # Copacabana
#
# # Rota mais curta (por distancia)
# rota = ox.shortest_path(G, orig, dest, weight="length")
# print(f"Rota: {len(rota)} segmentos")
#
# # Visualizar a rota no grafo
# fig, ax = ox.plot_graph_route(G, rota, node_size=0)
#
# # === PROXIMO PASSO: pesos customizados ===
# # Para priorizar ciclovias, adicione um atributo de peso a cada aresta:
# #
# # for u, v, data in G.edges(data=True):
# #     highway = data.get("highway", "")
# #     cycleway = data.get("cycleway", "")
# #     if highway == "cycleway" or cycleway in ["lane", "track"]:
# #         data["peso_ciclovia"] = data["length"] * 0.5  # bonus: ciclovia!
# #     else:
# #         data["peso_ciclovia"] = data["length"] * 2.0  # penalidade: sem ciclovia
# #
# # rota_ciclovia = ox.shortest_path(G, orig, dest, weight="peso_ciclovia")
# # Agora compare: rota por distancia vs rota priorizando ciclovia

## 9. Recursos Úteis

### Documentação

| Biblioteca | Link | Para que usar |
|---|---|---|
| osmnx | [osmnx.readthedocs.io](https://osmnx.readthedocs.io/) | Baixar e analisar dados do OpenStreetMap |
| GeoPandas | [geopandas.org](https://geopandas.org/en/stable/docs/user_guide.html) | Manipular dados geoespaciais em Python |
| Folium | [python-visualization.github.io/folium](https://python-visualization.github.io/folium/) | Criar mapas interativos |
| Shapely | [shapely.readthedocs.io](https://shapely.readthedocs.io/) | Operações geométricas (distância, interseção, buffer) |
| Leaflet.js | [leafletjs.com](https://leafletjs.com/) | Biblioteca JavaScript por trás do Folium |

### Dados abertos

| Fonte | Link | O que tem |
|---|---|---|
| OpenStreetMap Wiki | [wiki.openstreetmap.org/wiki/Key:cycleway](https://wiki.openstreetmap.org/wiki/Key:cycleway) | Documentação das tags de ciclovia |
| Data.Rio | [data.rio](https://www.data.rio/) | Dados abertos da prefeitura do RJ |
| GBFS Spec | [github.com/MobilityData/gbfs](https://github.com/MobilityData/gbfs) | Padrao de dados de bikesharing |

### Como contribuir

1. Faça um **fork** do repositório no GitHub
2. Crie uma branch para o seu desafio: `git checkout -b desafio-1-bike-itau`
3. Implemente a solução
4. Abra um **Pull Request** com descrição do que fez

Dúvidas? Fale com o professor ou supervisor do estágio.